# RAG System - Document Q&A & Chatbot

This notebook provides a complete RAG system with:
- Hybrid retrieval (BGE + BM25 + FAISS)
- Ollama/gemma4 generation
- Simple query interface
- Interactive chatbot

In [1]:
# Setup and Imports
import sys
import os
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path('.').absolute()))

from src.rag_system import RAGSystem
print('RAG System imported successfully')

W0506 22:11:16.576000 36636 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.



RAG System imported successfully


## Initialize RAG System

In [ ]:
# Initialize RAG system with source directory
SOURCE_DIR = Path('sources')

rag = RAGSystem(source_dir=SOURCE_DIR)
print('RAG System initialized')

# Check Ollama and preload model
from src.ollama_manager import OllamaManager

print("\n=== Ollama Status Check ===")
ollama_mgr = OllamaManager(
    base_url=rag.generator.config.base_url,
    model=rag.generator.config.model
)

status = ollama_mgr.get_status_summary()
if status["ollama_available"]:
    print("✅ Ollama is running")
    print(f"Available models: {status['available_models']}")

    if status["matching_model"]:
        print(f"✅ Model '{status['matching_model']}' is available")
        print(f"\n⏳ Loading model into memory...")
        success, elapsed = ollama_mgr.preload_model(status["matching_model"])
        if success:
            print(f"✅ Model ready ({elapsed:.1f}s)")
            rag.generator.config.model = status["matching_model"]
        else:
            print(f"⚠️ Model may not be fully loaded")
    else:
        print(f"⚠️ Configured model '{status['configured_model']}' not found")
else:
    print("❌ Ollama is NOT running")
    print("   Start with: `ollama serve`")

## Ingest Documents

In [3]:
# Ingest and index all documents
stats = rag.ingest_documents(SOURCE_DIR)
print(f"Ingested {stats['documents_loaded']} documents")
print(f"Created {stats['chunks_created']} chunks")
print(f"System stats: {rag.get_stats()}")

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Ingested 204 documents
Created 813 chunks
System stats: {'indexed': True, 'document_count': 813, 'model': 'gemma4:e4b', 'embedding_model': 'BAAI/bge-large-en-v1.5'}


## Query Interface

In [4]:
# Function to query the RAG system
def ask(question: str, top_k: int = 3):
    """Ask a question and display results."""
    result = rag.query(question, top_k=top_k)
    
    print(f"Question: {question}\n")
    print(f"Answer: {result.answer}\n")
    print("Sources:")
    for i, source in enumerate(result.sources, 1):
        print(f"  [{i}] {source['source']}, Page {source['page']} (score: {source['score']:.3f})")
    
    return result

### Example Queries

In [5]:
# Example: Ask about security
ask("What are the main security concerns for AI agents?")

Question: What are the main security concerns for AI agents?

Answer: AI security is centered on protecting systems from malicious threats [Source: Securing-Agentic-Applications-Guide-1.0.pdf, Page 70]. Furthermore, concerns include threat coverage gaps that can lead to defense gaps and inconsistent comparisons [Source: Securing-Agentic-Applications-Guide-1.0.pdf, Page 70]. It is important to note that AI security addresses malicious threats, which is distinct from AI safety, which ensures systems operate as intended without causing unintended harm [Source: Securing-Agentic-Applications-Guide-1.0.pdf, Page 70].

Sources:
  [1] Securing-Agentic-Applications-Guide-1.0.pdf, Page 70 (score: 0.028)
  [2] Cheat-Sheet-Agentic-AI-Solution-Landscape-Q226-1.pdf, Page 1 (score: 0.016)
  [3] Cheat-Sheet-Red-Teaming-AI-Solution-Landscape-Q226.pdf, Page 1 (score: 0.016)


RAGQueryResult(answer='AI security is centered on protecting systems from malicious threats [Source: Securing-Agentic-Applications-Guide-1.0.pdf, Page 70]. Furthermore, concerns include threat coverage gaps that can lead to defense gaps and inconsistent comparisons [Source: Securing-Agentic-Applications-Guide-1.0.pdf, Page 70]. It is important to note that AI security addresses malicious threats, which is distinct from AI safety, which ensures systems operate as intended without causing unintended harm [Source: Securing-Agentic-Applications-Guide-1.0.pdf, Page 70].', sources=[{'text': 'and threat coverage, leading to \ngaps in defense and inconsistent comparisons. The OWASP GenAI Red Teaming Guide has already curated a list of AI security benchmarks, focusing on key \nthreat vectors.\n...', 'source': 'Securing-Agentic-Applications-Guide-1.0.pdf', 'page': 70, 'section': '', 'score': 0.02844163539403516}, {'text': 'AI Security \nSolutions Landscape\nFor Agentic AI', 'source': 'Cheat-Shee

In [6]:
# Example: Ask about OWASP Top 10
ask("What does the OWASP Top 10 say about AI security?")

Question: What does the OWASP Top 10 say about AI security?

Answer: The OWASP Top 10 for Agentic Applications serves as a reference for security leaders and practitioners to understand and address the ten highest-impact threats in this area [Source: OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7]. This resource, part of the OWASP Gen AI Security Project, provides concise and actionable guidance using the standard OWASP Top 10 format [Source: OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7]. The coverage is focused on Agentic Apps, but the entries also reference other standards, including the OWASP Top 10 for LLM Applications and the OWASP AI Vulnerability Scoring System (AIVSS) for scoring [Source: OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7].

Sources:
  [1] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7 (score: 0.032)
  [2] OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 1 (score: 0.031)
  [3] OWASP-Top-10-for

RAGQueryResult(answer='The OWASP Top 10 for Agentic Applications serves as a reference for security leaders and practitioners to understand and address the ten highest-impact threats in this area [Source: OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7]. This resource, part of the OWASP Gen AI Security Project, provides concise and actionable guidance using the standard OWASP Top 10 format [Source: OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7]. The coverage is focused on Agentic Apps, but the entries also reference other standards, including the OWASP Top 10 for LLM Applications and the OWASP AI Vulnerability Scoring System (AIVSS) for scoring [Source: OWASP-Top-10-for-Agentic-Applications-2026-12.6-1.pdf, Page 7].', sources=[{'text': 'ications (or OWASP Agentic Top 10) serves as a compass and go-to \nreference for security leaders and practitioners to understand and address the Top 10 highest-impact \nthreats and begin their journey....', 'source': 'OWAS

## Interactive Chat

In [7]:
# Interactive chat function
def chat():
    """Interactive chat loop."""
    print('RAG Chatbot - Type your question or "quit" to exit\n')
    
    while True:
        question = input('You: ')
        if question.lower() in ['quit', 'exit', 'q']:
            print('Goodbye!')
            break
        
        result = rag.query(question)
        print(f"\nAssistant: {result.answer}\n")
        print('-' * 50)

In [8]:
# Start interactive chat (uncomment to use)
# chat()